In [0]:
ROOT = "/Workspace/Users/melaniepieterse1@gmail.com/Loan_Offer_Model"
CATALOG_PATH = f"{ROOT}/conf/base/catalog.yml"


In [0]:
import sys
sys.path.insert(0, ROOT)

from src.catalog_loader import Catalog
catalog = Catalog(spark, CATALOG_PATH)


In [0]:
customers_df = catalog.load("customer")
loans_df = catalog.load("loans")
transactions_df = catalog.load("transactions")

print(f"customers_df: {customers_df.count()} rows")
print(f"loans_df: {loans_df.count()} rows")
print(f"transactions_df: {transactions_df.count()} rows")


In [0]:
display(customers_df)
display(loans_df)
display(transactions_df)

Databricks filtered table. Run in Databricks to view.

In [0]:
%pip install ydata-profiling visions tzdata
import pandas as pd
from ydata_profiling import ProfileReport

In [0]:
# Evaluate the customer data profiles, identify outliers or missing values.
df_profile = ProfileReport(customers_df.toPandas(), minimal=True, title="Profiling Report", progress_bar=False, infer_dtypes=False)
profile_html = df_profile.to_html()
displayHTML(profile_html)

df_profile = ProfileReport(loans_df.toPandas(), minimal=True, title="Profiling Report", progress_bar=False, infer_dtypes=False)
profile_html = df_profile.to_html()
displayHTML(profile_html)

df_profile = ProfileReport(transactions_df.toPandas(), minimal=True, title="Profiling Report", progress_bar=False, infer_dtypes=False)
profile_html = df_profile.to_html()
displayHTML(profile_html)

In [0]:
from pyspark.sql.functions import *

# Use transactions data to identify potential income data for customers with missing income data. The income field is the monthly income data for each customer.

# Get the customer ID's that have missing income data. There are over 1,324 customers with missing income. 
monthly_income_to_inpute_df = customers_df.filter(col('INCOME').isNull()).select('ID').join(transactions_df.filter(col('AMOUNT') > 0), on = 'ID', how = 'left')
display(monthly_income_to_inpute_df)

# Use the transactional data to determine if there are a consistent income stream per customer ID  (postive transaction amounts) by looking at the statistics and any differences between the mode and max amounts
consistent_income = monthly_income_to_inpute_df.groupBy('ID').agg(count('*'), median(col('AMOUNT')).alias('median_AMOUNT'), avg(col('AMOUNT')).alias('average_AMOUNT'),mode(col('AMOUNT')).alias('mode_AMOUNT'),max(col('AMOUNT')).alias('max_AMOUNT')).withColumn('ADDITIONAL_INCOME', col('max_AMOUNT') - col('mode_AMOUNT'))
display(consistent_income)

# There are 344 customers  that show consistent income. This was deterimed if there were no difference between the mode and median amount, and if there were more than one income transaction.
inpute_consistent_income = consistent_income.filter(col('ADDITIONAL_INCOME')==0).select('ID','median_AMOUNT').withColumnRenamed('median_AMOUNT','INCOME')
display(inpute_consistent_income)

# Apply a filter to work with  the remaining customers with missing income data
filter_ids = [row.ID for row in inpute_consistent_income.select('ID').collect()]
monthly_income_to_inpute_df = monthly_income_to_inpute_df.filter(~col('ID').isin(filter_ids))
display(monthly_income_to_inpute_df)


#
additional_income = consistent_income.filter(col('ADDITIONAL_INCOME')!=0)
display(additional_income)


# Calculate how often the mode occurs for customers with additional income (ADDITIONAL_INCOME > 0)
def calculate_mode_frequency(additional_income_df, transactions_df):
    """
    For each customer in additional_income_df, compute how frequently the
    mode (most common) positive transaction amount appears in transactions_df.
    Returns a DataFrame with ID, mode_AMOUNT, and mode_count (how often the mode occurs).
    """
    # Get the IDs of customers with additional income
    income_ids = additional_income_df.filter(col('ADDITIONAL_INCOME') > 0).select('ID')

    # Join with positive transactions for those customers
    positive_txns = transactions_df.filter(col('AMOUNT') > 0).join(income_ids, on='ID', how='inner')

    # Count occurrences of each AMOUNT per customer
    amount_counts = positive_txns.groupBy('ID', 'AMOUNT').agg(count('*').alias('amount_count'))

    # Find the mode (AMOUNT with the highest count) and its frequency for each customer
    mode_info = amount_counts.groupBy('ID').agg(
        max(struct(col('amount_count'), col('AMOUNT'))).alias('max_struct')
    ).select(
        col('ID'),
        col('max_struct.AMOUNT').alias('mode_AMOUNT'),
        col('max_struct.amount_count').alias('mode_count')
    )

    return mode_info

mode_frequency_df = calculate_mode_frequency(additional_income, transactions_df)
display(mode_frequency_df)

# Compare mode count, to total count, to  distinct count

